In [1]:
!pip install xgboost
!pip install xlrd

In [2]:
import pandas as pd

#from pathlib import Path

In [3]:
from pathlib import Path

for path in Path("/kaggle/input").rglob("*"):
    if path.is_file():
        print(path)

df = pd.read_csv("/kaggle/input/datasets/ranaveerthanabuddhi/sdc-dataset/SDC_DATASET.xls")

/kaggle/input/datasets/ranaveerthanabuddhi/sdc-dataset/SDC_DATASET.xls


/tmp/ipykernel_51/3993515148.py:7: DtypeWarning: Columns (0,1,2,3,5,6,8,16,22,24,25,28,29,30,38,39,44,45,46,49,50,54) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/kaggle/input/datasets/ranaveerthanabuddhi/sdc-dataset/SDC_DATASET.xls")


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7877739 entries, 0 to 7877738
Data columns (total 55 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   Airport_Code             object 
 1   Amenity                  object 
 2   Astronomical_Twilight    object 
 3   Bump                     object 
 4   City                     object 
 5   Civil_Twilight           object 
 6   Country                  object 
 7   County                   object 
 8   Crossing                 object 
 9   Day_Of_Week              float64
 10  Description              object 
 11  Distance(mi)             float64
 12  Duration_Minutes         float64
 13  End_Lat                  float64
 14  End_Lng                  float64
 15  End_Time                 object 
 16  Give_Way                 object 
 17  Hour                     float64
 18  Humidity(%)              float64
 19  ID                       object 
 20  Is_Night                 float64
 21  Is_Weeke

In [5]:
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

In [6]:
selected_features = [
    "Start_Lat", "Start_Lng", "End_Lat", "End_Lng", "State",
    "Distance(mi)",
    "Temperature(F)", "Humidity(%)", "Pressure(in)",
    "Precipitation(in)", "Visibility(mi)", "Wind_Direction",
    "Wind_Speed(mph)", "Weather_Condition",
    "Hour", "Month", "Day_Of_Week", "Is_Weekend",
    "Rush_Hour", "Is_Night",
    "Amenity", "Crossing", "Junction", "Railway",
    "Stop", "Traffic_Signal"
]

X = df[selected_features].copy()
y = df["Severity"].copy()

In [7]:
start_time = pd.to_datetime(df["Start_Time"], format="mixed", errors="coerce")
valid_time = start_time.notna()

X["Hour"] = start_time.dt.hour
X["Month"] = start_time.dt.month
X["Day_Of_Week"] = start_time.dt.dayofweek

X["Is_Weekend"] = (
    (start_time.dt.dayofweek >= 5).astype("Int64").where(valid_time, pd.NA)
)
X["Is_Night"] = (
    ((start_time.dt.hour < 6) | (start_time.dt.hour >= 20))
    .astype("Int64").where(valid_time, pd.NA)
)
X["Rush_Hour"] = (
    (start_time.dt.hour.between(7, 9) | start_time.dt.hour.between(16, 18))
    .astype("Int64").where(valid_time, pd.NA)
)

print("Unparsed Start_Time values:", start_time.isna().sum())

Unparsed Start_Time values: 1


In [8]:
valid_target = y.notna()
X = X.loc[valid_target].copy()
y = y.loc[valid_target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 6302190
Test rows: 1575548


In [9]:
numeric_features = [
    "Start_Lat", "Start_Lng", "End_Lat", "End_Lng",
    "Distance(mi)", "Temperature(F)", "Humidity(%)",
    "Pressure(in)", "Precipitation(in)", "Visibility(mi)",
    "Wind_Speed(mph)", "Hour", "Month", "Day_Of_Week",
    "Is_Weekend", "Rush_Hour", "Is_Night"
]

categorical_features = [
    "State", "Wind_Direction", "Weather_Condition",
    "Amenity", "Crossing", "Junction", "Railway",
    "Stop", "Traffic_Signal"
]

for column in categorical_features:
    X_train[column] = X_train[column].fillna("Missing").astype(str)
    X_test[column] = X_test[column].fillna("Missing").astype(str)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (6302190, 270)
Processed test shape: (1575548, 270)


In [10]:
train_positions, _ = train_test_split(
    np.arange(len(y_train)),
    train_size=100_000,
    random_state=42,
    stratify=y_train
)

test_positions, _ = train_test_split(
    np.arange(len(y_test)),
    train_size=5_000,
    random_state=42,
    stratify=y_test
)

X_train_sampled = X_train_processed[train_positions]
y_train_sampled = y_train.iloc[train_positions]

X_test_sampled = X_test_processed[test_positions]
y_test_sampled = y_test.iloc[test_positions]

print("Sampled training shape:", X_train_sampled.shape)
print("Sampled test shape:", X_test_sampled.shape)

Sampled training shape: (100000, 270)
Sampled test shape: (5000, 270)


In [11]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        n_estimators=50,
        max_depth=4,
        learning_rate=0.1,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        n_jobs=-1
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=500
    )
}

accuracies = {}

for name, model in models.items():
    print("Training:", name)

    if name == "XGBoost":
        model.fit(X_train_sampled, y_train_sampled.astype(int) - 1)
        predictions = model.predict(X_test_sampled) + 1
    else:
        model.fit(X_train_sampled, y_train_sampled)
        predictions = model.predict(X_test_sampled)

    accuracies[name] = accuracy_score(y_test_sampled, predictions)

    print(f"{name} accuracy: {accuracies[name]:.4f}")
    print(classification_report(
        y_test_sampled,
        predictions,
        zero_division=0
    ))

best_model_name = max(accuracies, key=accuracies.get)
best_model = models[best_model_name]

print("Best model:", best_model_name)
print("Best accuracy:", round(accuracies[best_model_name], 4))

Training: XGBoost
XGBoost accuracy: 0.8142
              precision    recall  f1-score   support

         1.0       0.58      0.19      0.29        57
         2.0       0.82      0.98      0.89      3942
         3.0       0.72      0.21      0.33       847
         4.0       0.52      0.08      0.14       154

    accuracy                           0.81      5000
   macro avg       0.66      0.37      0.41      5000
weighted avg       0.79      0.81      0.77      5000

Training: KNN
KNN accuracy: 0.7930
              precision    recall  f1-score   support

         1.0       0.35      0.14      0.20        57
         2.0       0.84      0.93      0.88      3942
         3.0       0.51      0.35      0.42       847
         4.0       0.31      0.07      0.12       154

    accuracy                           0.79      5000
   macro avg       0.50      0.37      0.40      5000
weighted avg       0.76      0.79      0.77      5000

Training: Logistic Regression
Logistic Regression ac

In [12]:
model_data = {
    "model": best_model,
    "model_name": best_model_name,
    "preprocessor": preprocessor,
    "selected_features": selected_features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features
}

with open("SDC_PROJECT1.pkl", "wb") as file:
    pickle.dump(model_data, file)

print("Saved to SDC_PROJECT1.pkl")

Saved to SDC_PROJECT1.pkl


In [13]:
import os

print(os.path.exists("/kaggle/working/SDC_PROJECT1.pkl"))
print(os.listdir("/kaggle/working"))

True
['.virtual_documents', 'SDC_PROJECT1.pkl']


In [14]:
import pickle
import os

model_data = {
    "model": best_model,
    "model_name": best_model_name,
    "preprocessor": preprocessor,
    "selected_features": selected_features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features
}

save_path = "/kaggle/working/SDC_PROJECT1.pkl"

with open(save_path, "wb") as file:
    pickle.dump(model_data, file)

print("Saved:", os.path.exists(save_path))
print("File size:", os.path.getsize(save_path), "bytes")

Saved: True
File size: 357135 bytes
